In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
from sklearn import datasets
sklearn.set_config(display='text')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, Normalizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

## 앙상블 스태팅 알고리즘에 사용할 베이스 학습기와 메타 학습기 모델을 만들기 위해서 import 한다.

In [2]:
# 베이스 학습기
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
# 메타 학습기
from sklearn.ensemble import RandomForestClassifier

## 스태킹(stacking)

스태킹은 `베이스 학습기`와 `메타 학습기`로 구성되어 있고 베이스 학습기와 메타 학습기는 서포트 벡터 머신, 랜덤 포레스트 같은 학습 모델이다. `베이스 학습기가 먼저 학습`한 후 `메타 학습기`는 베이스 학습기의 예측 피쳐 데이터로 활용해서 `최종 예측`을 한다.

## 와인 데이터를 앙상블(스태킹) 알고리즘에 사용할 데이터를 준비한다.

In [3]:
raw_data = datasets.load_wine()

xData = raw_data.data
yData = raw_data.target

x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

## 스태킹 모델을 만들고 학습 시킨다.

스태킹 알고리즘을 사용하기 위해 StackingClassifier를 import 한다.

In [4]:
from sklearn.ensemble import StackingClassifier

베이스 학습기 모델을 만든다.

In [8]:
model_kn = KNeighborsClassifier(n_neighbors=27)
model_sv = SVC(probability=True, gamma=0.001)
model_dt = DecisionTreeClassifier(max_depth=8)
model_nb = GaussianNB()

메타 학습기 모델을 만든다.

In [19]:
model_rf = RandomForestClassifier(random_state=0, max_depth=4)

스태킹 모델을 만들고 학습시킨다.

`estimators`에 스태킹 알고리즘에 사용할 베이스 학습기 모델들을 리스트로 묶어서 지정한다.  
`final_estimator`에 스태킹 알고리즘에 사용할 메타 학습기 모델을 지정한다.

In [20]:
model = StackingClassifier(
    estimators=[('kn', model_kn), ('sv', model_sv), ('dt', model_dt), ('nb', model_nb)],
    final_estimator=model_rf
)
model.fit(x_train, y_train)

StackingClassifier(estimators=[('kn', KNeighborsClassifier(n_neighbors=27)),
                               ('sv', SVC(gamma=0.001, probability=True)),
                               ('dt', DecisionTreeClassifier(max_depth=8)),
                               ('nb', GaussianNB())],
                   final_estimator=RandomForestClassifier(max_depth=4,
                                                          random_state=0))

## 학습된 스태킹 모델을 테스트 데이터로 예측한다.

In [21]:
predict = model.predict(x_test)
print(predict)

[0 2 1 0 1 1 0 2 1 1 2 2 0 0 2 1 0 0 1 0 0 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1]


In [22]:
predict_proba = model.predict_proba(x_test)
print(predict_proba)

[[9.99545455e-01 4.54545455e-04 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]
 [1.66666667e-03 9.98333333e-01 0.00000000e+00]
 [9.99545455e-01 4.54545455e-04 0.00000000e+00]
 [0.00000000e+00 1.00000000e+00 0.00000000e+00]
 [9.09469697e-02 8.99053030e-01 1.00000000e-02]
 [9.99545455e-01 4.54545455e-04 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]
 [0.00000000e+00 1.00000000e+00 0.00000000e+00]
 [0.00000000e+00 1.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]
 [9.99545455e-01 4.54545455e-04 0.00000000e+00]
 [6.54280303e-01 3.15719697e-01 3.00000000e-02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]
 [0.00000000e+00 1.00000000e+00 0.00000000e+00]
 [9.99545455e-01 4.54545455e-04 0.00000000e+00]
 [9.99545455e-01 4.54545455e-04 0.00000000e+00]
 [1.42857143e-03 5.61071429e-01 4.37500000e-01]
 [9.99545455e-01 4.54545455e-04 0.00000000e+00]
 [6.19280303e-01 3.70719697e-01 1.000000

## 모델을 평가한다.

In [23]:
classification = classification_report(y_test, predict, target_names=raw_data.target_names)
print(classification)

              precision    recall  f1-score   support

     class_0       0.89      1.00      0.94        16
     class_1       1.00      0.90      0.95        21
     class_2       1.00      1.00      1.00         8

    accuracy                           0.96        45
   macro avg       0.96      0.97      0.96        45
weighted avg       0.96      0.96      0.96        45

